# 📊 Infosys Earnings Contradiction Detector

A RAG (Retrieval-Augmented Generation) system that detects contradictions in Infosys earnings call statements across quarterly reports (Q1, Q2, Q3 FY26).

## How It Works
1. **Ingest** — PDF transcripts are parsed and split into overlapping text chunks
2. **Embed** — Each chunk is converted into a semantic vector using `sentence-transformers`
3. **Store** — Vectors and metadata are stored in ChromaDB for similarity search
4. **Retrieve** — User queries are embedded and matched against stored chunks by quarter
5. **Detect** — An LLM compares retrieved chunks and identifies contradictions

## Stack
- `PyMuPDF` — PDF parsing
- `sentence-transformers` (all-MiniLM-L6-v2) — semantic embeddings
- `ChromaDB` — local vector database
- `Groq` (llama-3.1-8b-instant) — contradiction detection LLM
- `ipywidgets` — interactive UI

## Setup
Transcripts must be uploaded to Google Drive before running:
- `infosys_q1_fy26.pdf`
- `infosys_q2_fy26.pdf`
- `infosys_q3_fy26.pdf`

In [1]:
!pip install pymupdf sentence-transformers chromadb requests

## Step 1: Mount Google Drive
Transcripts and the embedding model are stored in Google Drive to persist across sessions.

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import os
model_path = "/content/drive/MyDrive/models/all-MiniLM-L6-v2"
files = os.listdir(model_path)
print(files)

# Check total size
total_size = 0
for f in files:
    fp = os.path.join(model_path, f)
    if os.path.isfile(fp):
        total_size += os.path.getsize(fp)
print(f"Total size: {total_size / 1024 / 1024:.1f} MB")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
['.cache', '1_Pooling', '.gitattributes', 'onnx', 'modules.json', 'README.md', 'config.json', 'data_config.json', 'config_sentence_transformers.json', 'model.safetensors', 'openvino', 'sentence_bert_config.json', 'special_tokens_map.json', 'pytorch_model.bin', 'tf_model.h5', 'tokenizer.json', 'tokenizer_config.json', 'train_script.py', 'vocab.txt', 'rust_model.ot']
Total size: 347.5 MB


In [3]:
import os
print('infosys_q1_fy26.pdf' in os.listdir('/content/drive/MyDrive'))
print('infosys_q2_fy26.pdf' in os.listdir('/content/drive/MyDrive'))
print('infosys_q3_fy26.pdf' in os.listdir('/content/drive/MyDrive'))

True
True
True


## Step 2: PDF Ingestion and Chunking

Transcripts are parsed page by page using PyMuPDF. Each page is:
1. Cleaned — boilerplate headers stripped with regex
2. Chunked — split into 200-word segments with 30-word overlap

**Why 200-word chunks?**
Smaller chunks produce more targeted embeddings. A 500-word chunk covering multiple topics produces a diluted vector that retrieves poorly. 200 words keeps each chunk semantically focused.

**Why 30-word overlap?**
A sentence split across two chunk boundaries loses context. Overlap ensures complete ideas aren't severed at chunk edges.

**Why skip pages 1-2?**
These pages contain only cover page content and analyst name lists — no substantive statements.

In [4]:
import fitz
import re

def clean_text(text):
    text = re.sub(r'External Document © \d{4} Infosys Limited \d+', '', text)
    return text.strip()

def extract_text_from_pdf(pdf_path):
    doc = fitz.open(pdf_path)
    pages = []
    for page_num in range(len(doc)):
        page = doc[page_num]
        text = page.get_text()
        if text.strip():
            pages.append((page_num + 1, text))
    return pages

def chunk_text(text, chunk_size=200, overlap=30):
    words = text.split()
    chunks = []
    start = 0
    while start < len(words):
        end = start + chunk_size
        chunk = " ".join(words[start:end])
        chunks.append(chunk)
        start += chunk_size - overlap
    return chunks

def ingest_transcript(pdf_path, company, quarter):
    pages = extract_text_from_pdf(pdf_path)
    all_chunks = []
    for page_num, page_text in pages:
        if page_num <= 2:
            continue
        page_text = clean_text(page_text)  # strip header noise
        chunks = chunk_text(page_text)
        for i, chunk in enumerate(chunks):
            if len(chunk.split()) < 60:
                continue
            all_chunks.append({
                "text": chunk,
                "company": company,
                "quarter": quarter,
                "page": page_num,
                "chunk_id": f"{company}_{quarter}_p{page_num}_c{i}"
            })
    return all_chunks

q1_chunks = ingest_transcript("/content/drive/MyDrive/infosys_q1_fy26.pdf", "Infosys", "Q1_FY26")
q2_chunks = ingest_transcript("/content/drive/MyDrive/infosys_q2_fy26.pdf", "Infosys", "Q2_FY26")
q3_chunks = ingest_transcript("/content/drive/MyDrive/infosys_q3_fy26.pdf", "Infosys", "Q3_FY26")
print(f"Q1 chunks: {len(q1_chunks)}")
print(f"Q2 chunks: {len(q2_chunks)}")
print(f"Q3 chunks: {len(q3_chunks)}")

Q1 chunks: 52
Q2 chunks: 33
Q3 chunks: 45


In [5]:
from sentence_transformers import SentenceTransformer

print("Loading model...")
model = SentenceTransformer('/content/drive/MyDrive/models/all-MiniLM-L6-v2')
print("Model loaded.")

test = model.encode(["hello world"])
print(f"Test embedding shape: {test.shape}")

Loading model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Model loaded.
Test embedding shape: (1, 384)


## Step 3: Embedding and Vector Storage

Each chunk is converted into a 384-dimensional vector using `all-MiniLM-L6-v2`.

**Why cosine similarity?**
Cosine similarity measures the angle between vectors rather than their magnitude. This is better for text — two chunks saying similar things in different lengths will have similar directions even if their magnitudes differ.

**Why EphemeralClient?**
ChromaDB's PersistentClient caused hanging issues in Colab. EphemeralClient stores vectors in memory for the session, which is sufficient since transcripts reload from Drive on each run.

In [6]:
import chromadb

client = chromadb.EphemeralClient()

try:
    client.delete_collection("earnings_transcripts")
    print("Deleted old collection.")
except:
    print("No existing collection to delete.")

def get_or_create_collection():
    return client.get_or_create_collection(
        name="earnings_transcripts",
        metadata={"hnsw:space": "cosine"}
    )

def embed_and_store(chunks):
    collection = get_or_create_collection()
    texts = [chunk["text"] for chunk in chunks]
    ids = [chunk["chunk_id"] for chunk in chunks]
    metadatas = [{
        "company": chunk["company"],
        "quarter": chunk["quarter"],
        "page": chunk["page"]
    } for chunk in chunks]

    print(f"Embedding {len(texts)} chunks...")
    embeddings = model.encode(texts, show_progress_bar=True)
    collection.add(
        documents=texts,
        embeddings=embeddings.tolist(),
        ids=ids,
        metadatas=metadatas
    )
    print(f"Done. Stored {len(texts)} chunks.")
    return collection

all_chunks = q1_chunks + q2_chunks + q3_chunks
print(f"Total chunks: {len(all_chunks)}")
collection = embed_and_store(all_chunks)

No existing collection to delete.
Total chunks: 130
Embedding 130 chunks...


Batches:   0%|          | 0/5 [00:00<?, ?it/s]

Done. Stored 130 chunks.


## Step 4: Semantic Retrieval

Retrieval uses cosine similarity between the query embedding and stored chunk embeddings. Metadata filters (`quarter`, `company`) ensure chunks are pulled from the correct transcript.

**Key design decision:** Quarter filtering is applied at the database level, not post-retrieval. This prevents a strong Q2 chunk from crowding out a weaker but relevant Q1 chunk when comparing quarters.

In [7]:
def retrieve_chunks(query, collection, quarter_filter=None, n_results=3):
    query_embedding = model.encode([query]).tolist()

    if quarter_filter:
        where_filter = {
            "$and": [
                {"company": {"$eq": "Infosys"}},
                {"quarter": {"$eq": quarter_filter}}
            ]
        }
    else:
        where_filter = {"company": {"$eq": "Infosys"}}

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        where=where_filter
    )

    chunks = []
    for i in range(len(results["documents"][0])):
        chunks.append({
            "text": results["documents"][0][i],
            "quarter": results["metadatas"][0][i]["quarter"],
            "page": results["metadatas"][0][i]["page"],
            "distance": results["distances"][0][i]
        })
    return chunks

In [8]:
!pip install groq

In [9]:
from groq import Groq
from google.colab import userdata

client_llm = Groq(api_key=userdata.get('GROQ_API_KEY'))

response = client_llm.chat.completions.create(
    model="llama-3.1-8b-instant",
    messages=[{"role": "user", "content": "What is 2 + 2? Answer in one word."}]
)
print(response.choices[0].message.content)

Four.


## Step 5: Contradiction Detection

Retrieved chunks from two quarters are passed to an LLM with a structured prompt. The prompt explicitly defines what counts as a contradiction (directional reversal, not just updated numbers) to reduce false positives.

**Known limitation:** Retrieval quality is moderate — cosine distances above 0.4 indicate weak semantic matches. A production system would benefit from hybrid search (BM25 + semantic) or a finance-domain embedding model.

In [10]:
def detect_contradiction(topic, collection, quarter_a, quarter_b):
    qa_chunks = retrieve_chunks(topic, collection, quarter_filter=quarter_a, n_results=2)
    qb_chunks = retrieve_chunks(topic, collection, quarter_filter=quarter_b, n_results=2)

    qa_context = "\n".join([f"[{quarter_a} Page {c['page']}]: {c['text']}" for c in qa_chunks])
    qb_context = "\n".join([f"[{quarter_b} Page {c['page']}]: {c['text']}" for c in qb_chunks])

    prompt = f"""You are a financial analyst comparing Infosys earnings call statements across quarters.

Topic: {topic}

{quarter_a} Statements:
{qa_context}

{quarter_b} Statements:
{qb_context}

Instructions:
- Identify if there is a clear contradiction between {quarter_a} and {quarter_b} on this topic.
- A contradiction means a clear reversal in direction or stance, not just updated numbers.
- If there is not enough information in either quarter, say so explicitly.
- Be specific and quote exact phrases that support your verdict.

Respond in exactly this format:
VERDICT: [CONTRADICTION FOUND / NO CONTRADICTION / INSUFFICIENT DATA]
REASON: [2-3 sentences explaining your verdict]
{quarter_a} CLAIM: [specific statement]
{quarter_b} CLAIM: [specific statement]"""

    response = client_llm.chat.completions.create(
        model="llama-3.1-8b-instant",
        messages=[{"role": "user", "content": prompt}]
    )
    return response.choices[0].message.content

## Step 6: Interactive UI

Select a topic and a quarter pair to compare. The detector returns:
- A verdict (CONTRADICTION FOUND / NO CONTRADICTION / INSUFFICIENT DATA)
- The reasoning behind the verdict
- The specific claims from each quarter
- The retrieved source chunks with similarity scores

**Example finding:** Querying "utilization and headcount capacity" (Q1 vs Q3) reveals a genuine strategic shift — Q1 describes peak utilization as a constraint requiring subcontractors, while Q3 shows deliberate headcount expansion to build future capacity.

In [11]:
import ipywidgets as widgets
from IPython.display import display, clear_output

title = widgets.HTML("<h2>📊 Infosys Earnings Contradiction Detector</h2><p>Compare statements across Q1, Q2, and Q3 FY26</p>")

topic_input = widgets.Text(
    placeholder="e.g. revenue growth guidance, hiring plans, operating margin",
    description="Topic:",
    layout=widgets.Layout(width="600px")
)

quarter_selector = widgets.Dropdown(
    options=[
        ("Q1 vs Q2", ("Q1_FY26", "Q2_FY26")),
        ("Q1 vs Q3", ("Q1_FY26", "Q3_FY26")),
        ("Q2 vs Q3", ("Q2_FY26", "Q3_FY26"))
    ],
    description="Compare:",
    layout=widgets.Layout(width="300px")
)

run_button = widgets.Button(
    description="Detect Contradiction",
    button_style="primary",
    layout=widgets.Layout(width="200px")
)

output_area = widgets.Output()

def on_button_click(b):
    topic = topic_input.value.strip()
    if not topic:
        with output_area:
            clear_output()
            print("Please enter a topic.")
        return

    quarter_a, quarter_b = quarter_selector.value

    with output_area:
        clear_output()
        print(f"Analyzing: '{topic}' | {quarter_a} vs {quarter_b}...")
        result = detect_contradiction(topic, collection, quarter_a, quarter_b)
        print("\n" + "="*60)
        print(result)
        print("="*60)

        print(f"\n--- Retrieved {quarter_a} Chunks ---")
        qa = retrieve_chunks(topic, collection, quarter_filter=quarter_a, n_results=2)
        for c in qa:
            print(f"[Page {c['page']} | Distance: {c['distance']:.3f}] {c['text'][:200]}")
            print()

        print(f"--- Retrieved {quarter_b} Chunks ---")
        qb = retrieve_chunks(topic, collection, quarter_filter=quarter_b, n_results=2)
        for c in qb:
            print(f"[Page {c['page']} | Distance: {c['distance']:.3f}] {c['text'][:200]}")
            print()

run_button.on_click(on_button_click)
display(title, topic_input, quarter_selector, run_button, output_area)

HTML(value='<h2>📊 Infosys Earnings Contradiction Detector</h2><p>Compare statements across Q1, Q2, and Q3 FY26…

Text(value='', description='Topic:', layout=Layout(width='600px'), placeholder='e.g. revenue growth guidance, …

Dropdown(description='Compare:', layout=Layout(width='300px'), options=(('Q1 vs Q2', ('Q1_FY26', 'Q2_FY26')), …

Button(button_style='primary', description='Detect Contradiction', layout=Layout(width='200px'), style=ButtonS…

Output()